# 04 · Explainable AI agent with one analysis tool

**Question:** Can an AI assistant explain the actual computed findings, clearly separate observations from uncertainty, and avoid medical diagnosis?

The system has exactly one allowlisted function: `analyze_segment(subject, start_seconds, duration_seconds)`. In the repository's optional Ollama integration, the LLM can ask for that function, receive a structured JSON result, and explain it. **The estimator is deterministic Python**, not an LLM guess.

The default example below is a **deterministic, non-LLM fallback** that lets you audit the entire tool result without an external service. A real local tool-calling LLM is optional and explicitly marked as such.

### 1. Environment and dataset configuration

**Default: real UCI data.** Set `PPG_DATA_ROOT` to the directory containing `S1/`, `S2/`, etc. Run from the project root (`jupyter lab`). If you cannot access the archive yet, `PPG_DEMO_MODE=1` enables **synthetic engineering-test data only**. Synthetic output must never be reported as dataset findings.

**Security:** loading the official `S1.pkl` deserializes Python pickle, which can run arbitrary code. Inspect the download and use an isolated, trusted environment. The agent cannot choose a data file or filesystem path.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')

HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE if (HERE / 'ppg_demo').is_dir() else HERE.parent
assert (PROJECT_ROOT / 'ppg_demo').is_dir(), 'Run Jupyter from the project root or notebooks/.'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = Path(os.getenv('PPG_DATA_ROOT', str(PROJECT_ROOT / 'data' / 'PPG_FieldStudy'))).expanduser()
DEMO_MODE = os.getenv('PPG_DEMO_MODE', '0').lower() in ('1', 'true', 'yes')
SUBJECT = os.getenv('PPG_SUBJECT', 'S1')

from ppg_demo.analysis import (load_subject, validate_subject, PPG_HZ, ACC_HZ,
    WIN_SECONDS, STEP_SECONDS, MIN_HZ, MAX_HZ)
from ppg_demo.synthetic import synthetic_record

validate_subject(SUBJECT)
if DEMO_MODE:
    record = synthetic_record(seconds=1100, subject=SUBJECT, bpm=90, movement_from_seconds=600)
    print('⚠ SYNTHETIC SMOKE TEST — NOT PPG-DaLiA MEASUREMENTS')
else:
    dataset_file = DATA_ROOT / SUBJECT / f'{SUBJECT}.pkl'
    if not dataset_file.is_file():
        raise FileNotFoundError(f'Cannot locate {dataset_file}. Extract the official UCI archive and set PPG_DATA_ROOT.')
    record = load_subject(str(dataset_file.resolve()))
    print(f'Official dataset, participant: {SUBJECT} | source: {dataset_file}')
print('BVP shape:', record.bvp.shape, '| ACC shape:', record.acc.shape, '| reference labels:', record.reference.shape)
plt.rcParams.update({'figure.figsize': (11, 4), 'axes.grid': True, 'grid.alpha': .18,
                     'axes.spines.top': False, 'axes.spines.right': False})

## Tool contract
### 2. Inspect the one tool available to the LLM

The tool accepts only a bounded segment and a validated subject ID. The LLM cannot set data paths or run arbitrary code. The reference is for evaluation only.

In [ ]:
import json
from IPython.display import display, Markdown
from ppg_demo.analysis import analyze_loaded_subject, analyze_segment
from ppg_demo.agent import TOOL, SYSTEM, template_explanation, ask_agent

print(json.dumps(TOOL, indent=2))
print('Agent requirements: no diagnosis; observations distinct from uncertainty; only tool-sourced numbers.')

## Demonstrate tool execution
### 3. Ask a natural-language question

Edit `QUESTION`, `START_SECONDS` and `DURATION_SECONDS` to investigate other regions. The function below is the same analysis as the previous notebooks, serialized without private raw waveforms.

In [ ]:
QUESTION = 'Analyse this recording segment. What did you find, and what are the limitations?'
START_SECONDS = 30
DURATION_SECONDS = 120

if DEMO_MODE:
    # Only for smoke tests; no official .pkl is involved.
    tool_result = analyze_loaded_subject(record, START_SECONDS, DURATION_SECONDS)
else:
    tool_result = analyze_segment(DATA_ROOT, SUBJECT, START_SECONDS, DURATION_SECONDS)

assert 'windows' not in tool_result and '_plot_signals' not in tool_result
print('Mode:', 'SYNTHETIC DEMO' if DEMO_MODE else 'OFFICIAL UCI DATA')
print('Question:', QUESTION)
print('Tool result (compact JSON):')
print(json.dumps(tool_result, indent=2, allow_nan=False))

### 4. Auditable explanation without a paid API

This is a useful fallback when no local LLM is available. **It is not an actual AI agent** and should not be described as one in the submission. The tool calling itself is real Python and can be inspected independently.

In [ ]:
deterministic_answer = template_explanation(tool_result)
display(Markdown('**Deterministic fallback (not an LLM):**\n\n' + deterministic_answer))

## Optional: actual local LLM tool calling
### 5. One Ollama tool call (no paid API)

Install [Ollama](https://ollama.com/), run `ollama pull qwen3:4b`, and set `PPG_RUN_OLLAMA=1` to enable this cell. The model must support tool calling. The flow is **user → Ollama → `analyze_segment` → Ollama → explanation**. It is intentionally skipped in synthetic smoke-test mode.

The code prints the *exact tool result* alongside the answer to make hallucinations visible. Do not state this was tested unless you actually run it.

In [ ]:
RUN_OLLAMA = os.getenv('PPG_RUN_OLLAMA', '0').lower() in ('1', 'true', 'yes')
if RUN_OLLAMA and not DEMO_MODE:
    model_name = os.getenv('PPG_OLLAMA_MODEL', 'qwen3:4b')
    llm_answer, actual_tool_result = ask_agent(
        QUESTION, DATA_ROOT, subject=SUBJECT,
        start_seconds=START_SECONDS, duration_seconds=DURATION_SECONDS,
        model=model_name)
    display(Markdown('**Actual local LLM response:**\n\n' + llm_answer))
    print('Tool output used by the LLM:')
    print(json.dumps(actual_tool_result, indent=2, allow_nan=False))
else:
    print('LLM tool-call demonstration NOT RUN.' if not RUN_OLLAMA else
          'Real LLM tool call skipped: synthetic testing is not a real participant analysis.')
    print('Set PPG_RUN_OLLAMA=1 and provide official data to try a local tool-calling model.')

## Verification and limitations

For any actual LLM run, manually compare the answer to the printed JSON. Check that it:

1. Reports no fabricated BPM, MAE, segment lengths, or activity labels.
2. Distinguishes computed observations from hypothetical causes (e.g., potential motion interference).
3. Describes spectral concentration / acceleration overlap as **heuristics**, not confidence probabilities.
4. Makes no diagnoses or individualized clinical claims.
5. Does not claim generalization from one participant or short segment.

If the model ignores the tool, the helper raises an error rather than generating an unsupported analysis. State clearly whether the actual model path was executed.

**Next →** complete `REPORT.md` with verified numerical results, unsuccessful attempts, resources used, and genuine time spent.